# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library. We load the Croissant schema, review data structure, and perform exploratory data analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset with Croissant
dataset = mlc.Dataset(url)

# Access metadata via `.metadata` attributes
print(f"Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. List all record sets, their `@id`s, and the `@id`s of fields contained in each record set.


In [ ]:
# List all available record sets with their @id and human-readable name
print("Available Record Sets:")
if hasattr(dataset.metadata, 'record_sets'):
    record_sets = dataset.metadata.record_sets
else:
    record_sets = [rs for rs in dataset.metadata.recordSet] if hasattr(dataset.metadata, 'recordSet') else []
rs_ids = []

for rs in dataset.record_sets:
    print(f"  @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")
    rs_ids.append(rs['@id'])

# For each record set, list fields (@id and name)
for rs in dataset.record_sets:
    print(f"\nFields for Record Set @id: {rs['@id']} - {rs.get('name', '(no name)')}")
    for field in rs['fields']:
        print(f"    Field @id: {field['@id']}, name: {field.get('name', '(no name)')}")

## 3. Data Extraction
Load data from available record set(s) into pandas DataFrame(s) for analysis. Use the record set and field `@id`s from the overview. For this dataset, we will demonstrate loading all rows from the main record set.

In [ ]:
dataframes = {}
loaded_rs = []

# Loop through all record sets and load into DataFrames
for rs in dataset.record_sets:
    rs_id = rs['@id']
    print(f"Loading records for record set @id={rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        loaded_rs.append(rs_id)
        print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns for {rs_id}")
        print(f"Sample columns: {df.columns[:10].tolist()}")
    except Exception as e:
        print(f"Failed to load record set {rs_id}: {e}\n")

# For demonstration, print the first 5 rows and columns for the first available record set
if loaded_rs:
    main_rs = loaded_rs[0]
    print(f"\nColumns for main record set {main_rs}:\n", dataframes[main_rs].columns.tolist())
    dataframes[main_rs].head()
else:
    print('No record sets could be loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps such as filtering, normalization and grouping for a selected numeric field. For this example, we'll use the GAD-7 score if available, referencing by its `@id` field (usually like 'gad7_total' or similar; refer to the field list above). You may need to adjust field names as per the printed columns above.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if loaded_rs:
    df = dataframes[main_rs].copy()
    # Guess likely GAD-7 or PHQ-9 total @id
    available_cols = df.columns.tolist()
    gad7_candidates = [c for c in available_cols if 'gad' in c.lower() and 'total' in c.lower()]
    phq9_candidates = [c for c in available_cols if 'phq' in c.lower() and 'total' in c.lower()]
    numeric_field_id = None
    if gad7_candidates:
        numeric_field_id = gad7_candidates[0]
    elif phq9_candidates:
        numeric_field_id = phq9_candidates[0]
    else:
        # Fallback: first numeric column
        for c in available_cols:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field_id = c
                break
    print(f"Numeric field chosen for EDA: {numeric_field_id}")

    # Use a threshold for filtering (e.g., mild anxiety)
    threshold = 10
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize column
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a demographic field, e.g., 'gender' or similar
        group_field = None
        for demo_col in ['gender', 'sex', 'village', 'level_of_education']:
            if demo_col in available_cols:
                group_field = demo_col
                break

        if group_field:
            grouped_df = (
                filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            )
            print(f"\nMeans of {numeric_field_id} grouped by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No appropriate numeric field found for EDA.")
else:
    print('Cannot run EDA: no loaded record set.')

## 5. Visualization
Visualize numeric field distribution and group means. We use Seaborn for convenience.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if loaded_rs and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print('No suitable numeric field or record set for visualization.')

## 6. Conclusion
In this notebook, we loaded the FAIR² Kilifi Mental Health Survey dataset, explored its structure through the Croissant schema, and applied basic exploratory data analysis on demographic and mental health fields. For more comprehensive insights, repeat analysis across additional fields or adjust filtering/grouping to match research questions.

**Note**: All dataset fields and entities are referenced by their Croissant `@id` where applicable. For in-depth analysis, consult the [Croissant schema JSON-LD](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) for field semantics and details.